### Output Parsers

Output parsers help LLMs give output in a structured format like TypedDict, JSON, or Pydantic model.

These are same as with_structured_output function in langchain, the only difference is that Output parsers can work with both type of LLMs: which "can" or "can not" give structured outputs 


4 most used Output Parsers:
- String Output Parser
- JSON Output Parser
- Structured Output Parser
- Pydantic Output Parser

### 1. String Output Parser
One of the most basic output parser in langchaing. Takes the output of LLM and return the content as string 

Then why are these important when we can just use "output.content"?   
The answer is: To implement this thing in chains 

Below is comparison between both- A. Without using StrOutputParser,  B. With StrOutputParser

In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

#Load .env file
load_dotenv()

True

without String Output Parser

In [16]:
llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task='text-generation',
    temperature=0.5
)

model = ChatHuggingFace(llm = llm)

#Prompt1 -> Generate text
template1 = PromptTemplate(
    template="Write a report of 10-20 lines on {topic}",
    input_variables=['topic']
)

#Prompt2 -> Give summary
template2 = PromptTemplate(
    template="Write a summary of 5 pointers on given text: {text}",
    input_variables= ['text']
)

In [17]:
prompt1 = template1.invoke({'topic':'Black Holes'})
result1 = model.invoke(prompt1)

prompt2 = template2.invoke({'text':result1.content})
result2 = model.invoke(prompt2)

In [18]:
print(result2.content)

In this report, we will explore the phenomenon of black holes and their impact on the universe. We will delve into their properties, including their size, mass, energy and momentum, and their rate of matter ejection. We will also discuss the role of black holes in shaping the evolution of the universe and their potential to lead to new discoveries and insights.

Black Holes: A Cosmic Object

Black holes are one of the most fascinating and mysterious objects in the universe. They are the result of the collapse of massive stars, and they have a singularity at their center that is so dense that nothing can escape its gravitational pull. Black holes are a crucial part of the cosmic history, and understanding them is crucial to our understanding of the universe.

Size, Mass, Energy, and Momentum

A black hole has a size, mass, energy, and momentum. The size of a black hole is determined by the mass of the black hole. The larger the mass, the larger the radius of the black hole. The mass of 

with StrOutputParser

In [19]:
parser = StrOutputParser()
chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic':'Black Holes'})

In [20]:
print(result)

In this report, we will explore the properties and behavior of black holes, their formation and evolution, and their impact on the universe. Black holes are one of the most mysterious and fascinating objects in the universe. They are the result of the collapse of massive stars, and their gravitational pull is so intense that even light cannot escape from their vicinity. Black holes are the source of cosmic radiation, and their gravitational pull can destroy planets or even entire galaxies.

Part 1: Black Holes: Their Formation and Evolution

Black holes are formed through the collapse of massive stars, which are the most dense objects in the universe. The collapse of a star occurs when the gravitational force of the star exceeds the gravitational force of its surroundings. When this happens, the star collapses to a point smaller than its radius, and the matter at the center of the star is compressed into a singularity. The singularity is the point of no return, where the gravitational 

### JSON Outpur Parser

In [10]:
#Changed the llm cuz TinyLlama wasnt working properly

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task='text-generation',
    temperature=0.5
)

model = ChatHuggingFace(llm = llm)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser


json_parser = JsonOutputParser()

template = PromptTemplate(
    template="Give acutal name, age, gender of this fictional character : {character}.\n {format_instructions}",
    input_variables=['character'],
    partial_variables={'format_instructions': json_parser.get_format_instructions()}
)

#Why is format_instructions given as partial_variable? Because it fills the template before invoking the prompt 

#Just to show prompt
prompt = template.invoke({'character':'Spider Man'})

chain = template | model | parser
result = chain.invoke({'character':'Spider Man'})

In [42]:
print(prompt.text)
print("_"*60, "\n")
print(result)
print("_"*60, "\n")
print(type(result))

Give acutal name, age, gender of this fictional character : Spider Man.
 Return a JSON object.
____________________________________________________________ 

{"name": "Peter Parker", "age": 15, "gender": "male"}
____________________________________________________________ 

<class 'str'>


### StructureOutputParser

This to enforce any pre-difined schema to the output

In [ ]:
schema = [
    ResponseSchema(name='Name', type='string', description="Actual name of the this character"),
    ResponseSchema(name='Age', type='int', description="Age of this character"),
    ResponseSchema(name='Gender', type='string', description='Gender of this character')
]

structured_parser = StructuredOutputParser.from_response_schemas(response_schemas=schema)

template = PromptTemplate(
    template="Give acutal name, age, gender of this fictional character : {character}.\n {format_instructions}",
    input_variables=['character'],
    partial_variables={'format_instructions': structured_parser.get_format_instructions()}
)

prompt = template.invoke({'character':'Spider Man'})
print(prompt.text)

chain = template | model | structured_parser
result = chain.invoke({'character':'Spider Man'})

Give acutal name, age, gender of this fictional character : Spider Man.
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"Name": string  // Actual name of the this character
	"Age": int  // Age of this character
	"Gender": string  // Gender of this character
}
```


In [18]:
print(result)
print(type(result))

{'Name': 'Peter Parker', 'Age': 16, 'Gender': 'Male'}
<class 'dict'>


### Pydantic Output Parser
Pydantic output parser is a structured output parser that uses pydantic models to enfore schema validation in LLM responses

- Type Safety (by type coercion)
- Data validation 
- Strict Schema enforcement

In [12]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

#Create pydantic model
class Character(BaseModel):
    name: str = Field(..., description="Give the actual name of Character")
    age: int = Field(..., description="Give age of character")
    gender: Literal['Male','Female'] = Field(..., description="Give the gender of character")

pydantic_parser = PydanticOutputParser(pydantic_object=Character)

template = PromptTemplate(
    template="Give acutal name, age, gender of this fictional character : {character}.\n {format_instructions}",
    input_variables=['character'],
    partial_variables={'format_instructions': pydantic_parser.get_format_instructions()}
)

prompt = template.invoke({'character':'spider man'})
print(prompt.text)

chain = template | model | pydantic_parser
result = chain.invoke({'character':'spider man'})

Give acutal name, age, gender of this fictional character : spider man.
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Give the actual name of Character", "title": "Name", "type": "string"}, "age": {"description": "Give age of character", "title": "Age", "type": "integer"}, "gender": {"description": "Give the gender of character", "enum": ["Male", "Female"], "title": "Gender", "type": "string"}}, "required": ["name", "age", "gender"]}
```


In [15]:
print(result)
print(type(result))

name='Peter Parker' age=28 gender='Male'
<class '__main__.Character'>
